# Explainable Cyber Threat Intelligence (CTI) Agent — trained with GRPO

In [ ]:
# Goal: teach Qwen2.5-3B-Instruct to read a CTI text snippet and output the
# correct MITRE ATT&CK Technique ID inside <answer> tags, preceded by
# step-by-step reasoning inside <reasoning> tags.
#
# Algorithm: GRPO (Group Relative Policy Optimization) via TRL.
#
#   Why GRPO instead of PPO?
#   ------------------------
#   PPO needs a separate Value/Critic model (typically as large as the policy)
#   to estimate the expected reward of every token. That doubles GPU memory and
#   struggles with credit assignment on long reasoning chains.
#
#   GRPO removes the Critic entirely. For each prompt it samples a *group* of G
#   completions, scores each one with cheap, deterministic reward functions
#   (rule-based Python — no neural network), and computes each completion's
#   advantage RELATIVE to the group:
#
#           A_i = (r_i - mean(r)) / std(r)
#
#   Completions that beat the group average are reinforced; those below it are
#   pushed down. No Critic => ~half the memory => fits a single 16GB Kaggle GPU.
#
# Target hardware: Kaggle T4 (compute 7.5, fp16 — NO bf16).
# =============================================================================

# Installs  (run these in the first Kaggle cell, then RESTART the session)

In [1]:
# Unsloth pins a compatible trl + vllm for you; version drift between
# unsloth / trl / vllm / torch is the #1 cause of import errors on Kaggle.
# Kaggle already ships a working torch build, so do NOT reinstall torch.
#
!pip install -q unsloth vllm
!pip install -q --no-deps "trl==0.15.2" datasets

#
# If you hit a vLLM/torch mismatch, prefer the single line:
!pip install -q "unsloth[kaggle-new]" vllm
# This installs the libraries quietly, hiding the Kaggle dependency warnings
!pip install -q transformers peft trl datasets bitsandbytes accelerate > /dev/null 2>&1
# -----------------------------------------------------------------------------

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 1.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 24.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 103.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 92.3 MB/s eta 0:00:00:00

In [ ]:
import os

# IMPORTANT: must be set BEFORE `import unsloth`.
# On Kaggle T4 (compute 7.5), vLLM tries to JIT-compile FlashInfer's sampling
# kernel and the link step fails with "/usr/bin/ld: cannot find -lcuda" (the
# CUDA driver stub isn't on the linker path in the Kaggle image). Installing
# ninja does NOT fix it — ninja runs fine; the linker is what fails. Disabling
# FlashInfer makes vLLM fall back to its native Triton top-k/top-p sampler,
# which needs no compilation and works on the T4.
os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"

import ast
import re

import torch
from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

# -----------------------------------------------------------------------------
# 1. Model initialization — Unsloth FastLanguageModel + vLLM fast inference
# -----------------------------------------------------------------------------
# NOTE: we load the 7B via Unsloth's FastLanguageModel (NOT plain
# AutoModelForCausalLM) because the whole GRPO pipeline needs 4-bit weights +
# vLLM fast generation + LoRA training, which the vanilla transformers path
# from the model card cannot provide.
#
# HEADS-UP on hardware: Qwen2-7B in 4-bit + vLLM KV-cache + LoRA training +
# sampling G completions is TIGHT on a single 16GB T4. It generally fits with
# the reduced settings below, but if you OOM, in order: (a) drop num_generations
# AND per_device_train_batch_size to 2 together, (b) lower max_completion_length
# to 200, (c) lower gpu_memory_utilization. On P100 (no bf16, weaker vLLM) prefer
# the 3B model instead.
MAX_SEQ_LENGTH = 1024        # prompt (CTI snippet) + completion budget
LORA_RANK = 16               # smaller rank than the 3B run to save 7B memory
                             #   (must stay <= max_lora_rank below)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,           # 4-bit QLoRA base — mandatory to fit 7B on a T4
    fast_inference=True,         # route generation through vLLM.
                                 #   GRPO's bottleneck is sampling G completions
                                 #   per prompt; vLLM makes that ~10x faster.
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.55, # vLLM KV-cache share. Kept modest so the 7B
                                 #   LoRA forward/backward still fits in 16GB.
    dtype=torch.float16,         # T4 has NO bf16 — force fp16 everywhere.
)

# Attach LoRA adapters. We only train these; the 4-bit base stays frozen.
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
        "gate_proj", "up_proj", "down_proj",      # MLP projections
    ],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient variant
    random_state=3407,
)

# -----------------------------------------------------------------------------
# 2. Dataset — tumeteor/Security-TTP-Mapping (CTI text -> MITRE technique IDs)
# -----------------------------------------------------------------------------
# Schema: column "text1" (CTI snippet) and "labels" (a *stringified* Python list
# such as "['T1057']"). Rows may carry multiple technique IDs, and IDs may be
# sub-techniques (e.g. "T1059.001").
#
# The system prompt below forces the exact output contract we reward for.
SYSTEM_PROMPT = (
    "You are a cyber threat intelligence analyst. You are given a threat "
    "intelligence text snippet describing adversary behavior. Identify the "
    "single most relevant MITRE ATT&CK technique.\n"
    "Think step-by-step inside <reasoning> tags and output the exact MITRE "
    "ATT&CK ID (e.g. T1059 or T1059.001) inside <answer> tags.\n"
    "Respond in EXACTLY this format and nothing else:\n"
    "<reasoning>\n...your step-by-step analysis...\n</reasoning>\n"
    "<answer>T####</answer>"
)


def _parse_labels(raw):
    """Turn a stringified label list into a clean list of technique IDs."""
    if isinstance(raw, list):
        ids = raw
    else:
        try:
            ids = ast.literal_eval(str(raw))
        except (ValueError, SyntaxError):
            # sometimes labels arrive as a bare string like "T1057"
            ids = [raw]
        if not isinstance(ids, list):
            ids = [ids]
    # normalize: uppercase, strip, drop empties
    return [str(i).strip().upper() for i in ids if str(i).strip()]


def _to_grpo_row(example):
    """Map a raw dataset row into the conversational format GRPOTrainer wants."""
    ids = _parse_labels(example["labels"])
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"CTI snippet:\n{example['text1']}"},
        ],
        # extra columns are forwarded verbatim to the reward functions as kwargs
        "ground_truth": ids,
    }


def build_dataset():
    """Load Security-TTP-Mapping; fall back to a tiny synthetic set if offline."""
    try:
        raw = load_dataset("tumeteor/Security-TTP-Mapping", split="train")
        ds = raw.map(_to_grpo_row, remove_columns=raw.column_names)
        # drop rows whose labels failed to parse into at least one ID
        ds = ds.filter(lambda r: len(r["ground_truth"]) > 0)
        print(f"[data] loaded Security-TTP-Mapping: {len(ds)} rows")
        return ds
    except Exception as e:  # noqa: BLE001 — Kaggle may block the download
        print(f"[data] load failed ({e!r}); using synthetic fallback set")
        samples = [
            ("Process 'cmd.exe' spawned 'powershell.exe -enc JABzAD0A...'.", "T1059.001"),
            ("The malware queried the registry to enumerate running processes.", "T1057"),
            ("A scheduled task was created to run the payload every 10 minutes.", "T1053.005"),
            ("Credentials were dumped from LSASS memory using a custom tool.", "T1003.001"),
            ("The sample established a TLS connection to 198.51.100.34:443 for C2.", "T1071.001"),
            ("Files in the user directory were encrypted and a ransom note dropped.", "T1486"),
            ("The binary copied itself to the Startup folder for persistence.", "T1547.001"),
            ("It executed 'whoami' and 'ipconfig /all' to profile the host.", "T1082"),
        ]
        rows = [
            _to_grpo_row({"text1": text, "labels": f"['{tid}']"})
            for text, tid in samples
        ]
        return Dataset.from_list(rows)


dataset = build_dataset()

# -----------------------------------------------------------------------------
# 3. Reward functions
# -----------------------------------------------------------------------------
# GRPOTrainer calls each reward function with (prompts, completions, **kwargs).
# `completions` is a list (one per sampled generation) of message lists in
# conversational format, so the assistant text is completions[i][0]["content"].
# Any extra dataset column — here `ground_truth` — is forwarded as a same-length
# list keyword argument. Each function returns a list[float] of rewards.

# Strict full-output contract: <reasoning>...</reasoning><answer>...</answer>
_STRICT_FORMAT = re.compile(
    r"^\s*<reasoning>.*?</reasoning>\s*<answer>.*?</answer>\s*$",
    re.DOTALL,
)
# Pull the text inside the FIRST <answer>...</answer>
_ANSWER = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.DOTALL)
# A MITRE technique ID, optionally with a sub-technique (T1059 / T1059.001)
_TID = re.compile(r"T\d{4}(?:\.\d{3})?", re.IGNORECASE)


def _completion_text(completion):
    """Extract the assistant string from one conversational completion."""
    return completion[0]["content"]


def format_reward_func(prompts, completions, **kwargs):
    """1.0 if the completion strictly matches the required XML format, else 0.0."""
    rewards = []
    for c in completions:
        text = _completion_text(c)
        rewards.append(1.0 if _STRICT_FORMAT.match(text) else 0.0)
    return rewards


def _extract_id(text):
    """Return the normalized technique ID inside <answer>, or None."""
    m = _ANSWER.search(text)
    if not m:
        return None
    hit = _TID.search(m.group(1))
    return hit.group(0).upper() if hit else None


def correctness_reward_func(prompts, completions, ground_truth=None, **kwargs):
    """+2.0 if the predicted ID exactly matches ANY ground-truth ID, else -1.0.

    Multi-label aware: a row may list several valid techniques. Exact match is
    required, so a sub-technique like T1059.001 does NOT satisfy a bare T1059
    (and vice versa) unless that exact ID is in the label list.
    """
    if ground_truth is None:
        ground_truth = [[] for _ in completions]
    rewards = []
    for c, truth in zip(completions, ground_truth):
        pred = _extract_id(_completion_text(c))
        truth_set = {t.strip().upper() for t in truth}
        rewards.append(2.0 if pred is not None and pred in truth_set else -1.0)
    return rewards


# Optional soft nudge: partial credit for producing the tags at all. This eases
# the cold-start phase before the model reliably emits the full format. Tunable —
# remove it from `reward_funcs` below if you want a purely strict signal.
def soft_format_reward_func(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        text = _completion_text(c)
        score = 0.0
        if "<reasoning>" in text and "</reasoning>" in text:
            score += 0.25
        if "<answer>" in text and "</answer>" in text:
            score += 0.25
        rewards.append(score)
    return rewards


# -----------------------------------------------------------------------------
# 4. GRPO configuration
# -----------------------------------------------------------------------------
# num_generations (G) is the group size scored per prompt. G=4 is the practical
# ceiling for a 7B model in 4-bit on a 16GB T4 (G=8 will OOM during generation);
# drop to G=2 if you still run out of memory.
#
# IMPORTANT divisibility rule (TRL will assert this):
#   (per_device_train_batch_size * gradient_accumulation_steps * num_gpus)
#   must be divisible by num_generations.
# Here 4 * 1 * 1 = 4, and 4 % 4 == 0. If you hit OOM, drop BOTH num_generations
# and per_device_train_batch_size together (e.g. to 2) so they stay divisible.
NUM_GENERATIONS = 4

training_args = GRPOConfig(
    output_dir="/kaggle/working/outputs",
    # --- GRPO group sampling ---
    num_generations=NUM_GENERATIONS,
    max_prompt_length=640,
    max_completion_length=256,
    use_vllm=True,               # generate with the vLLM engine loaded above
    # --- batching (see divisibility note) ---
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    # --- precision: T4 => fp16 only, bf16 unavailable ---
    fp16=True,
    bf16=False,
    # --- optimization ---
    learning_rate=5e-6,          # RL fine-tuning likes a small LR
    optim="adamw_8bit",          # 8-bit optimizer to save memory
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.1,
    max_grad_norm=0.1,
    # --- logging / duration / saving ---
    logging_steps=1,
    max_steps=250,               # demo length; raise (or use num_train_epochs)
                                 #   for a real run once rewards look healthy.
    save_steps=250,
    report_to="none",
)

# -----------------------------------------------------------------------------
# 5. Trainer
# -----------------------------------------------------------------------------
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,       # 0.0 / 1.0   — strict XML contract
        correctness_reward_func,  # -1.0 / +2.0 — right MITRE ID (dominant signal)
        soft_format_reward_func,  # 0.0..0.5    — cold-start scaffolding (optional)
    ],
    args=training_args,
    train_dataset=dataset,
)

# -----------------------------------------------------------------------------
# 6. Train + save the LoRA adapters
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    trainer.train()

    # Save just the trained LoRA adapters to the Kaggle working dir (downloadable).
    model.save_lora("/kaggle/working/grpo_cti_lora")
    model.save_pretrained("/kaggle/working/grpo_cti_adapters")
    tokenizer.save_pretrained("/kaggle/working/grpo_cti_adapters")
    print("[done] LoRA adapters saved to /kaggle/working/")

    # To ship a standalone 16-bit model instead of adapters, use:
    #   model.save_pretrained_merged(
    #       "/kaggle/working/grpo_cti_merged", tokenizer, save_method="merged_16bit")


# -----------------------------------------------------------------------------
# Reward smoke test (run BEFORE the expensive train() to sanity-check scoring).
# Copy this block into a scratch cell — it needs no GPU.
# -----------------------------------------------------------------------------
# def _msg(text):
#     return [{"role": "assistant", "content": text}]
#
# good = _msg("<reasoning>cmd spawns encoded powershell => command interpreter"
#             "</reasoning>\n<answer>T1059.001</answer>")
# bad_fmt = _msg("The answer is T1059.001")
# wrong = _msg("<reasoning>looks like discovery</reasoning>\n<answer>T1082</answer>")
# gt = [["T1059.001"]]
#
# assert format_reward_func(None, [good[:1]]) == [1.0]
# assert format_reward_func(None, [bad_fmt[:1]]) == [0.0]
# assert correctness_reward_func(None, [good[:1]], ground_truth=gt) == [2.0]
# assert correctness_reward_func(None, [wrong[:1]], ground_truth=gt) == [-1.0]
# print("reward functions OK")